In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans



In [ ]:
catlog = pd.read_csv("Audible_Catlog.csv")
features = pd.read_csv("Audible_Catlog_Advanced_Features.csv")

print("catlog shape:", catlog.shape)
print("features shape:", features.shape)

catlog.head()
features.head()

print(catlog.columns)
print(features.columns)

In [ ]:
df = pd.merge(catlog, features, on="Book Name", how="inner")

print("Duplicate rows:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("After removal:", df.shape)

df['Number of Reviews_x'] = df['Number of Reviews_x'].fillna('')
df['Number of Reviews_y'] = df['Number of Reviews_y'].fillna('')
df['Price_x'] = df['Price_x'].fillna('')
df['Description'] = df['Description'].fillna('')

print(df.isnull().sum())

In [ ]:
df.columns = df.columns.str.strip().str.lower()

df.rename(columns={
    'book name': 'title',
    'author_x': 'author',
    'author_y': 'author_dup',
    'rating_x': 'rating',
    'rating_y': 'rating_dup',
    'number of reviews_x': 'reviews',
    'number of reviews_y': 'reviews_dup',
    'price_x': 'price',
    'price_y': 'price_dup',
    'ranks and genre': 'genre'
}, inplace=True)

text_cols = ['title', 'author', 'author_dup', 'description', 'genre']

for col in text_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

df.drop(columns=['author_dup', 'rating_dup', 'reviews_dup', 'price_dup'], inplace=True)

print(df.columns)
df.head()



In [ ]:
print(df.duplicated(subset=["title", "author"]).sum())

In [ ]:
duplicates = df[df.duplicated(subset=["title", "author"], keep=False)]
duplicates[["title", "author", "rating"]].sort_values(by=["title", "author"]).head(20)

In [ ]:
df = df.drop_duplicates(subset=["title", "author"]).reset_index(drop=True)
print(df.duplicated(subset=["title", "author"]).sum())
df.to_csv("cleaned_books_data.csv", index=False)


In [ ]:
def extract_category(text):
    if pd.isna(text):
        return "unknown"
    
    # Extract text after "in"
    match = re.search(r'in (.*?)(\(|$)', text)
    
    if match:
        return match.group(1).lower().strip()
    else:
        return "unknown"

df['clean_genre'] = df['genre'].apply(extract_category)

df['clean_genre'].head(10)

In [ ]:
top_genres = df['clean_genre'].value_counts().head(10)

plt.figure()
top_genres.plot(kind='bar')
plt.title("Top 10 Genres")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
df['content'] = (
    df['title'] + " " +
    df['author'] + " " +
    df['description']
)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)  # remove special chars
    return text

df['content'] = df['content'].apply(clean_text)

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(df['content'])

print(tfidf_matrix.shape)

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

def recommend_books(book_title):
    book_title = book_title.lower()
    
    matches = df[df['title'].str.contains(book_title)]
    
    if matches.empty:
        suggestions = df[df['title'].str.contains(book_title[:3])]
        return suggestions['title'].head(5).values
    
    idx = matches.index[0]
    
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]
    
    book_indices = [i[0] for i in scores]
    
    return df[['title', 'author', 'rating']].iloc[book_indices]

recommend_books("rich")

In [ ]:
wcss = []

for k in range(1, 16):
    model = KMeans(n_clusters=k, random_state=42)
    model.fit(tfidf_matrix)
    wcss.append(model.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1,16), wcss, marker='o')
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

silhouette_scores = []

for k in range(2, 11):  # Starts from 2 because silhouette score cannot be computed for K=1
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(tfidf_matrix)

    score = silhouette_score(tfidf_matrix, labels)
    silhouette_scores.append(score)

    print(f"K={k}, Silhouette Score={score:.4f}")

In [ ]:
import matplotlib.pyplot as plt

k_values = range(2, 11)

plt.figure(figsize=(8,5))
plt.plot(k_values, silhouette_scores, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")
plt.grid(True)
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)

df['cluster'] = kmeans.fit_predict(tfidf_matrix)

df[['title', 'cluster']].head()

def recommend_by_cluster(book_title):
    book_title = book_title.lower()
    
    if book_title not in df['title'].values:
        return "Book not found"
    
    cluster_id = df[df['title'] == book_title]['cluster'].values[0]
    
    return df[df['cluster'] == cluster_id]['title'].head(5)

In [ ]:
def recommend_content(title, k=5):
    title = title.lower()
    matches = df[df['title'].str.contains(title)]
    if matches.empty:
        return []
    
    idx = matches.index[0]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:k+1]
    
    idxs = [i for i,_ in scores]
    return df[['title','author','rating']].iloc[idxs].reset_index(drop=True)

In [ ]:
def recommend_cluster(title, k=5):
    title = title.lower()
    matches = df[df['title'].str.contains(title)]
    if matches.empty:
        return []
    
    idx = matches.index[0]
    cluster_id = df.loc[idx, 'cluster']
    
    recs = df[df['cluster'] == cluster_id]
    recs = recs[recs.index != idx]  # remove same book
    
    return recs[['title','author','rating']].head(k).reset_index(drop=True)

In [ ]:
def recommend_hybrid(title, k=5):

    content = recommend_content(title, k)
    cluster = recommend_cluster(title, k)

    hybrid = content.head(3).copy()

    for _, row in cluster.iterrows():
        if row["title"] not in hybrid["title"].values:
            hybrid = pd.concat(
                [hybrid, row.to_frame().T],
                ignore_index=True
            )

        if len(hybrid) >= k:
            break

    return hybrid.reset_index(drop=True)

In [ ]:
def precision_at_k(recommended, relevant, k=5):
    recommended = list(recommended)[:k]
    relevant = set(relevant)
    
    hits = len([r for r in recommended if r in relevant])
    return hits / k

In [ ]:
def recall_at_k(recommended, relevant, k=5):
    recommended = list(recommended)[:k]
    relevant = set(relevant)
    
    hits = len([r for r in recommended if r in relevant])
    return hits / len(relevant) if len(relevant) > 0 else 0

In [ ]:
book = "rich"

# recommendations
content_rec = recommend_content(book)['title']
cluster_rec = recommend_cluster(book)['title']
hybrid_rec = recommend_hybrid(book)['title']

# define relevant set (same cluster)
idx = df[df['title'].str.contains(book)].index[0]
cluster_id = df.loc[idx, 'cluster']
relevant_books = df[df['cluster'] == cluster_id]['title']

content_prec = precision_at_k(content_rec, relevant_books)
cluster_prec = precision_at_k(cluster_rec, relevant_books)
hybrid_prec = precision_at_k(hybrid_rec, relevant_books)

content_recall = recall_at_k(content_rec, relevant_books)
cluster_recall = recall_at_k(cluster_rec, relevant_books)
hybrid_recall = recall_at_k(hybrid_rec, relevant_books)


In [ ]:
models = ["Content-Based", "Cluster-Based", "Hybrid"]
precision = [content_prec, cluster_prec, hybrid_prec]
recall = [content_recall, cluster_recall, hybrid_recall]

f1 = [
    2 * p * r / (p + r) if (p + r) > 0 else 0
    for p, r in zip(precision, recall)
]

comparison_df = pd.DataFrame({
    "Model": models,
    "Precision@5": precision,
    "Recall@5": recall,
    "F1@5": f1
})

comparison_df